# 12 · Path-Dependent Options

Exotic options require path simulation. Finstack exposes Monte Carlo GBM models for Asian and lookback payoffs with a consistent JSON builder.

### Learning Objectives
- Build equity markets for exotic options (discount curve, spot, dividend yield, vol surface)
- Configure arithmetic/geometric Asian options via the builder interface
- Price lookback options with the Monte Carlo engine and inspect the resulting PV

In [ ]:
from datetime import date

from finstack import Money
from finstack.core.currency import USD
from finstack.core.market_data import MarketContext
from finstack.core.market_data.scalars import MarketScalar
from finstack.core.market_data.surfaces import VolSurface
from finstack.core.market_data.term_structures import DiscountCurve
from finstack.valuations.instruments import AsianOption, LookbackOption
from finstack.valuations.pricer import create_standard_registry

val_date = date(2025, 1, 1)
market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD.SOFR",
        val_date,
        [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (2.0, 0.90)],
    )
)
market.insert_surface(
    VolSurface(
        "ACME.VOL",
        expiries=[0.5, 1.0],
        strikes=[140.0, 150.0, 160.0, 170.0],
        grid=[
            [0.25, 0.23, 0.21, 0.23],
            [0.27, 0.25, 0.23, 0.25],
        ],
    )
)
market.insert_price("ACME", MarketScalar.price(Money(150.0, USD)))
market.insert_price("ACME.DIV", MarketScalar.unitless(0.01))
registry = create_standard_registry()


## 1. Arithmetic Asian Call
Arithmetic averaging uses discrete fixing dates. The builder accepts fixing date lists plus payoff details.

In [ ]:
fixings = [
    date(2025, 3, 1),
    date(2025, 6, 1),
    date(2025, 9, 1),
    date(2025, 12, 1),
]
asian_call = AsianOption.builder(
    instrument_id="ASIAN-CALL",
    ticker="ACME",
    strike=150.0,
    expiry=date(2025, 12, 31),
    fixing_dates=fixings,
    notional=Money(10000.0, USD),
    discount_curve="USD.SOFR",
    spot_id="ACME",
    vol_surface="ACME.VOL",
    averaging_method="arithmetic",
    option_type="call",
    div_yield_id="ACME.DIV",
)
asian_res = registry.price(asian_call, "monte_carlo_gbm", market, as_of=val_date)
print("Asian call PV:", asian_res.value)


## 2. Geometric Asian Put
Switch the averaging method to `geometric` and set option type to `put`.

In [ ]:
geo_fixings = [
    date(2025, 1, 31),
    date(2025, 2, 28),
    date(2025, 3, 31),
    date(2025, 4, 30),
    date(2025, 5, 31),
    date(2025, 6, 30),
]
asian_put = AsianOption.builder(
    instrument_id="ASIAN-PUT",
    ticker="ACME",
    strike=160.0,
    expiry=date(2025, 6, 30),
    fixing_dates=geo_fixings,
    notional=Money(50000.0, USD),
    discount_curve="USD.SOFR",
    spot_id="ACME",
    vol_surface="ACME.VOL",
    averaging_method="geometric",
    option_type="put",
    div_yield_id="ACME.DIV",
)
geo_res = registry.price(asian_put, "monte_carlo_gbm", market, as_of=val_date)
print("Geometric Asian put PV:", geo_res.value)


## 3. Fixed-Strike Lookback Call
Lookback payoffs depend on the maximum/minimum path level. Use the builder to toggle lookback type.

In [ ]:
lookback_call = LookbackOption.builder(
    instrument_id="LOOKBACK",
    ticker="ACME",
    strike=150.0,
    option_type="call",
    lookback_type="fixed_strike",
    expiry=date(2025, 12, 31),
    notional=Money(150000.0, USD),
    discount_curve="USD.SOFR",
    spot_id="ACME",
    vol_surface="ACME.VOL",
    div_yield_id="ACME.DIV",
)
lookback_res = registry.price(lookback_call, "monte_carlo_gbm", market, as_of=val_date)
print("Lookback call PV:", lookback_res.value)


## Summary
- Path-dependent payoffs use builder patterns to capture fixing schedules and payoff variants.
- Monte Carlo GBM models power both Asian and lookback pricers via `monte_carlo_gbm`.
- Use the same market context as vanilla options (discount + spot + vol + dividend).